# 🏨 HotelRank: Sentiment Analysis & SAW Ranking

Notebook ini akan mendemonstrasikan secara interaktif (step-by-step) bagaimana **HotelRank** bekerja. Mulai dari melatih model Machine Learning untuk Sentiment Analysis, menggunakan model tersebut untuk memprediksi sentimen ulasan hotel, hingga menerapkan algoritma cerdas **SAW (Simple Additive Weighting)** untuk memeringkat hotel terbaik.

## 📚 Tahap 1: Persiapan Library
Kita akan mengimpor library utama seperti pandas, scikit-learn, dan mensimulasikan data hotel.

In [65]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

## 🧠 Tahap 2: Simulasi Sentiment Analysis
Di sini kita akan membuat contoh model sederhana menggunakan 10 teks Bahasa Inggris dan Indonesia untuk mendemonstrasikan bagaimana Machine Learning belajar memahami sentimen.

In [66]:
# 1. 10 Contoh Data Latih Sederhana
data = {
    "text": [
        "Kamar sangat bersih dan nyaman.", 
        "Pelayanan buruk, air panas mati.",
        "Fasilitas biasa saja, lumayan untuk harga segitu.",
        "Great hotel, love the breakfast!",
        "Dirty room and rude staff.",
        "Lokasi sangat strategis, dekat dengan stasiun kereta.",
        "AC bocor dan bau rokok di kamar non-smoking.",
        "Not bad, but could be better for the price.",
        "Makanan hambar, tidak sebanding dengan harga.",
        "Proses check-in sangat cepat dan resepsionis sangat membantu."
    ],
    "label": [
        "Positive", "Negative", "Neutral", "Positive", "Negative", 
        "Positive", "Negative", "Neutral", "Negative", "Positive"
    ]
}
df_train = pd.DataFrame(data)
display(df_train)

# 2. Vektorisasi Teks (TF-IDF)
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df_train["text"])
y = df_train["label"]

# 3. Latih Model Regresi Logistik
model = LogisticRegression()
model.fit(X, y)

print("\n✅ Model berhasil dilatih dengan 10 sampel!")

# 4. Tes Prediksi Baru
test_text = ["Kasurnya empuk dan staf ramah"]
test_vec = vectorizer.transform(test_text)
prediction = model.predict(test_vec)
print(f"\n🧪 Teks Uji: '{test_text[0]}' -> Prediksi Sentimen: {prediction[0]}")

,text,label
0,Kamar sangat bersih dan nyaman.,Positive
1,"Pelayanan buruk, air panas mati.",Negative
2,"Fasilitas biasa saja, lumayan untuk harga segitu.",Neutral
3,"Great hotel, love the breakfast!",Positive
4,Dirty room and rude staff.,Negative
5,"Lokasi sangat strategis, dekat dengan stasiun ...",Positive
6,AC bocor dan bau rokok di kamar non-smoking.,Negative
7,"Not bad, but could be better for the price.",Neutral
8,"Makanan hambar, tidak sebanding dengan harga.",Negative
9,Proses check-in sangat cepat dan resepsionis s...,Positive



✅ Model berhasil dilatih dengan 10 sampel!

🧪 Teks Uji: 'Kasurnya empuk dan staf ramah' -> Prediksi Sentimen: Positive


## 🧮 Tahap 3: Algoritma SAW (Simple Additive Weighting)
Setelah kita mendapatkan sentimen dari model AI, kita akan menggunakan metode **SAW** untuk merangking hotel berdasarkan 3 kriteria Benefit:
- **C1 (Fasilitas Hotel)**: Bobot 0.45 -- nilai `c1_weighted` maks <= **0.45**
- **C2 (Score Sentiment)**: Bobot 0.30 -- nilai `c2_weighted` maks <= **0.30**
- **C3 (Rata-rata Rating)**: Bobot 0.25 -- nilai `c3_weighted` maks <= **0.25**

### 3.1. Matriks Keputusan (Nilai Mentah)
Nilai C2 (0-100) dan C3 (0-10) masih dalam skala aslinya. Penyeragaman dilakukan di tahap normalisasi.

In [67]:
# 1. 10 Contoh Data Hotel (Simulasi JSON hasil Scraping)
hotels = [
    {"name": "Hotel Indah Jaya", "favoriteFeatures": ["Free Wi-Fi in all rooms!", "Breakfast [free]", "Air conditioning", "Swimming pool"], "averageSentimentScore": 85.0, "averageRating": 8.8},
    {"name": "Penginapan Sederhana", "favoriteFeatures": ["Free Wi-Fi in all rooms!", "Air conditioning"], "averageSentimentScore": 60.0, "averageRating": 6.5},
    {"name": "Luxury Resort Spa", "favoriteFeatures": ["Free Wi-Fi in all rooms!", "Restaurant [halal]", "Room service", "Airport transfer", "Massage", "Elevator"], "averageSentimentScore": 92.5, "averageRating": 9.4},
    {"name": "Budget Inn City Center", "favoriteFeatures": ["Free Wi-Fi in all rooms!", "Check-in/out [express]"], "averageSentimentScore": 50.5, "averageRating": 5.8},
    {"name": "Grand Emerald Suites", "favoriteFeatures": ["Free Wi-Fi in all rooms!", "Breakfast [free]", "Fitness center", "Swimming pool", "Shuttle service", "Elevator", "Shared lounge/TV area"], "averageSentimentScore": 88.0, "averageRating": 9.0},
    {"name": "Sea View Beach Hotel", "favoriteFeatures": ["Free Wi-Fi in all rooms!", "Restaurants", "Private beach", "Room service", "Balcony/terrace"], "averageSentimentScore": 78.5, "averageRating": 8.2},
    {"name": "Transit Hotel Airport", "favoriteFeatures": ["Free Wi-Fi in all rooms!", "Airport transfer", "Room service [24-hour]", "Cash withdrawal"], "averageSentimentScore": 65.0, "averageRating": 7.0},
    {"name": "Boutique Art Hotel", "favoriteFeatures": ["Free Wi-Fi in all rooms!", "Coffee shop", "Garden", "Safety deposit boxes"], "averageSentimentScore": 82.0, "averageRating": 8.5},
    {"name": "Eco Lodge Resort", "favoriteFeatures": ["Restaurant [halal]", "Hiking", "Tours", "Garden"], "averageSentimentScore": 75.0, "averageRating": 8.0},
    {"name": "The Presidential Palace", "favoriteFeatures": ["Free Wi-Fi in all rooms!", "Restaurant [halal]", "Room service", "Massage", "Sauna", "Elevator", "Concierge", "Smoking area"], "averageSentimentScore": 96.0, "averageRating": 9.8}
]

# 2. Fungsi Hitung Fasilitas (C1)
def calculate_c1(features):
    score = 0.0
    features = [f.lower() for f in features]
    if any("wifi" in f or "wi-fi" in f for f in features): score += 0.10 # Internet
    if any("breakfast" in f or "restaurant" in f for f in features or "coffee shop" in f): score += 0.15 # F&B
    if any("room service" in f or "air conditioning" in f or "balcony" in f for f in features): score += 0.10 # Comfort
    if any("airport" in f or "shuttle" in f for f in features): score += 0.05 # Transport
    if any("pool" in f or "massage" in f or "beach" in f or "fitness" in f or "sauna" in f or "garden" in f or "hiking" in f or "tours" in f for f in features): score += 0.03 # Recreation
    if any("elevator" in f or "check-in" in f or "lounge" in f or "cash" in f or "safety" in f or "concierge" in f or "smoking" in f for f in features): score += 0.02 # Other
    return score

# Ekstrak Nilai Mentah (X)
for h in hotels:
    h["c1_raw"] = calculate_c1(h["favoriteFeatures"])
    h["c2_raw"] = h["averageSentimentScore"]
    h["c3_raw"] = h["averageRating"]

df_hotels = pd.DataFrame(hotels)
display(df_hotels[["name", "c1_raw", "c2_raw", "c3_raw"]])

,name,c1_raw,c2_raw,c3_raw
0,Hotel Indah Jaya,0.38,85.0,8.8
1,Penginapan Sederhana,0.20,60.0,6.5
2,Luxury Resort Spa,0.45,92.5,9.4
3,Budget Inn City Center,0.12,50.5,5.8
4,Grand Emerald Suites,0.35,88.0,9.0
5,Sea View Beach Hotel,0.38,78.5,8.2
6,Transit Hotel Airport,0.27,65.0,7.0
7,Boutique Art Hotel,0.15,82.0,8.5
8,Eco Lodge Resort,0.18,75.0,8.0
9,The Presidential Palace,0.40,96.0,9.8


### 3.2. Normalisasi Matrix & Perhitungan V-Score (Skor Akhir)
Karena ketiga kriteria bersifat **Benefit** (Semakin tinggi semakin bagus), maka kita menormalisasinya dengan rumus:

$$ R_{ij} = \frac{X_{ij}}{Max_j} $$

Lalu untuk mendapatkan *Ranking Score* (V-Score), kita mengalikan R dengan Bobot W:
$$ V_i = \sum W_j \times R_{ij} $$

In [68]:
# 1. Cari Nilai Max tiap kriteria
max_c1 = df_hotels['c1_raw'].max()
max_c2 = df_hotels['c2_raw'].max()
max_c3 = df_hotels['c3_raw'].max()

# 2. Bobot Kriteria
w_c1, w_c2, w_c3 = 0.45, 0.30, 0.25

# 3. Normalisasi (R) -- skala 0 hingga 1
df_hotels['c1_norm'] = df_hotels['c1_raw'] / max_c1
df_hotels['c2_norm'] = df_hotels['c2_raw'] / max_c2
df_hotels['c3_norm'] = df_hotels['c3_raw'] / max_c3

# 4. Nilai Preferensi per Kriteria (W x R) -- tidak boleh melebihi bobot masing-masing
df_hotels['c1_weighted'] = w_c1 * df_hotels['c1_norm']  # Maks: 0.45
df_hotels['c2_weighted'] = w_c2 * df_hotels['c2_norm']  # Maks: 0.30
df_hotels['c3_weighted'] = w_c3 * df_hotels['c3_norm']  # Maks: 0.25

# 5. V-Score Akhir (jumlah semua weighted) -- maks: 1.0
df_hotels['saw_score'] = df_hotels['c1_weighted'] + df_hotels['c2_weighted'] + df_hotels['c3_weighted']
df_hotels['rank'] = df_hotels['saw_score'].rank(ascending=False).astype(int)

# 6. Tampilkan Hasil Akhir dengan kolom weighted (bukan raw)
df_final = df_hotels.sort_values('rank')[['rank', 'name', 'saw_score', 'c1_weighted', 'c2_weighted', 'c3_weighted']]
display(df_final)

,rank,name,saw_score,c1_weighted,c2_weighted,c3_weighted
2,1,Luxury Resort Spa,0.978858,0.45,0.289062,0.239796
9,2,The Presidential Palace,0.950000,0.40,0.300000,0.250000
0,3,Hotel Indah Jaya,0.870115,0.38,0.265625,0.224490
4,4,Grand Emerald Suites,0.854592,0.35,0.275000,0.229592
5,5,Sea View Beach Hotel,0.834496,0.38,0.245312,0.209184
6,6,Transit Hotel Airport,0.651696,0.27,0.203125,0.178571
7,7,Boutique Art Hotel,0.623087,0.15,0.256250,0.216837
8,8,Eco Lodge Resort,0.618457,0.18,0.234375,0.204082
1,9,Penginapan Sederhana,0.553316,0.20,0.187500,0.165816
3,10,Budget Inn City Center,0.425772,0.12,0.157812,0.147959
